<a href="https://colab.research.google.com/github/MartVASS/MaskArchitectureAnomaly_CourseProject/blob/main/notebooks/Fine_tuning_COCO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/MartVASS/MaskArchitectureAnomaly_CourseProject.git
%cd MaskArchitectureAnomaly_CourseProject/eomt

fatal: destination path 'MaskArchitectureAnomaly_CourseProject' already exists and is not an empty directory.
/content/MaskArchitectureAnomaly_CourseProject/eomt


In [2]:
!pip install -r requirements.txt

In [3]:
!pip uninstall wandb
!pip install wandb

!wandb login

Found existing installation: wandb 0.19.10
Uninstalling wandb-0.19.10:
  Would remove:
    /usr/local/bin/wandb
    /usr/local/bin/wb
    /usr/local/lib/python3.12/dist-packages/package_readme.md
    /usr/local/lib/python3.12/dist-packages/wandb-0.19.10.dist-info/*
    /usr/local/lib/python3.12/dist-packages/wandb/*
Proceed (Y/n)? y
  Successfully uninstalled wandb-0.19.10
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 79.5 MB/s eta 0:00:00
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vasseurmartin2003 (vasseurmartin2003-politecnico-di-torino) to https://api.wandb.ai. Use `wandb login --relogin` to f

In [4]:
import os
import re

file_path = "/content/MaskArchitectureAnomaly_CourseProject/eomt/training/mask_classification_panoptic.py"

# Reset file to add patch
!git checkout -- {file_path}

with open(file_path, "r") as f:
    code = f.read()

# new code with flatten and filtration of valid indices [0, 18]
patched_eval_step = """    def eval_step(
        self,
        batch,
        batch_idx=None,
        log_prefix=None,
    ):
        import torch
        import torch.nn.functional as F
        from torchmetrics.classification import MulticlassJaccardIndex

        # Init mIoU calculator (19 classes)
        if not hasattr(self, "semantic_miou_metric"):
            self.semantic_miou_metric = MulticlassJaccardIndex(num_classes=19).to(self.device)

        # Init mapping tensor COCO -> Cityscapes
        if not hasattr(self, "coco_to_cityscapes_map"):
            user_mapping = {100: 0, 123: 1, 129: 2, 109: 3, 110: 3, 111: 3, 112: 3, 131: 3, 117: 4, 9: 6, 11: 7, 116: 8, 125: 8, 88: 8, 102: 9, 103: 9, 119: 10, 0: 11, 2: 13, 7: 14, 5: 15, 6: 16, 3: 17, 1: 18}

            # We use 255 for non mapped value
            full_map = torch.ones(134, dtype=torch.long) * 255
            for coco_id, city_id in user_mapping.items():
                full_map[coco_id] = city_id
            self.coco_to_cityscapes_map = full_map.to(self.device)

        imgs, targets = batch

        img_sizes = [img.shape[-2:] for img in imgs]
        transformed_imgs = self.resize_and_pad_imgs_instance_panoptic(imgs)
        mask_logits_per_layer, class_logits_per_layer = self(transformed_imgs)

        is_crowds = [target["is_crowd"] for target in targets]
        targets = self.to_per_pixel_targets_panoptic(targets)

        for i, (mask_logits, class_logits) in enumerate(
            list(zip(mask_logits_per_layer, class_logits_per_layer))
        ):
            mask_logits = F.interpolate(mask_logits, self.img_size, mode="bilinear")
            mask_logits = self.revert_resize_and_pad_logits_instance_panoptic(
                mask_logits, img_sizes
            )
            preds = self.to_per_pixel_preds_panoptic(
                mask_logits,
                class_logits,
                self.stuff_classes,
                self.mask_thresh,
                self.overlap_thresh,
            )

            # --- Semantic mIoU calculation ---
            if i == len(mask_logits_per_layer) - 1:
                for p, t in zip(preds, targets):
                    # 1. Conversion and flattened en 1D
                    mapped_preds = self.coco_to_cityscapes_map[p.long()].view(-1)
                    clean_targets = t.long().view(-1)

                    # 2. Security filter : we keep only targets valid in Citscapes [0, 18]
                    # We get rid off -1, 255, and non-mapped COCO prediction.
                    valid_mask = (clean_targets >= 0) & (clean_targets < 19) & (mapped_preds != 255)

                    # 3. Input valid pixel to torchmetrics
                    if valid_mask.any():
                        self.semantic_miou_metric.update(mapped_preds[valid_mask], clean_targets[valid_mask])

            self.update_metrics_panoptic(preds, targets, is_crowds, i)

        if batch_idx % 20 == 0:
            current_miou = self.semantic_miou_metric.compute()
            print(f" -> Image {batch_idx:03d}/500 | current mIoU Semantic Cityscapes (Zero-Shot) : {current_miou*100:.2f}%")
"""

pattern = r"    def eval_step\(.*?self\.update_metrics_panoptic\(preds, targets, is_crowds, i\)"
code_modified, count = re.subn(pattern, patched_eval_step, code, flags=re.DOTALL)

if count > 0:
    with open(file_path, "w") as f:
        f.write(code_modified)
    print("File repatched sucessfully !")
else:
    print("Error during repatch...")

File repatched sucessfully !


In [5]:
!cat configs/dinov2/cityscapes/semantic/eomt_base_640.yaml

trainer:
  max_epochs: 107
  logger:
    class_path: lightning.pytorch.loggers.wandb.WandbLogger
    init_args:
      resume: allow
      project: "eomt"
      name: "cityscapes_semantic_eomt_base_640"
model:
  class_path: training.mask_classification_semantic.MaskClassificationSemantic
  init_args:
    attn_mask_annealing_enabled: True
    attn_mask_annealing_start_steps: [3317, 8292, 13268]
    attn_mask_annealing_end_steps: [6634, 11609, 16585]
    network:
      class_path: models.eomt.EoMT
      init_args:
        num_q: 100
        num_blocks: 3
        encoder:
          class_path: models.vit.ViT
          init_args:
            backbone_name: vit_base_patch14_reg4_dinov2
data:
  class_path: datasets.cityscapes_semantic.CityscapesSemantic

In [6]:
!grep -r "lr\|learning_rate" training/mask_classification_semantic.py

        lr: float = 1e-4,
        llrd: float = 0.8,
        llrd_l2_enabled: bool = True,
        lr_mult: float = 1.0,
            lr=lr,
            llrd=llrd,
            llrd_l2_enabled=llrd_l2_enabled,
            lr_mult=lr_mult,


In [7]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [25]:
import os
import torch, gc

data_path = "/content/drive/MyDrive/Cityscapes"
path_to_model = "/content/drive/MyDrive/COCO/eomt_coco.bin"
finetuned_path = "/content/drive/MyDrive/finetuned_cityscapes"
eomt_path = "/content/MaskArchitectureAnomaly_CourseProject/eomt"

filepath = "/content/MaskArchitectureAnomaly_CourseProject/eomt/training/mask_classification_semantic.py"

with open(filepath, "r") as f:
    content = f.read()

# Comment plot bloc
content = content.replace(
    """            if batch_idx == 0:
                self.plot_semantic(
                    imgs[0], targets[0], logits[0], log_prefix, i, batch_idx
                )""",
    """            # if batch_idx == 0:
            #     self.plot_semantic(
            #         imgs[0], targets[0], logits[0], log_prefix, i, batch_idx
            #     )"""
)

with open(filepath, "w") as f:
    f.write(content)

print("Patch applied")

!grep -n "plot_semantic" /content/MaskArchitectureAnomaly_CourseProject/eomt/training/mask_classification_semantic.py
gc.collect()
torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

%cd {eomt_path}

Patch appliqué ✓
114:            #     self.plot_semantic(
/content/MaskArchitectureAnomaly_CourseProject/eomt


In [ ]:
!python3 main.py fit \
  -c configs/dinov2/cityscapes/semantic/eomt_base_640.yaml \
  --data.path /content/drive/MyDrive/Cityscapes \
  --model.ckpt_path /content/drive/MyDrive/COCO/eomt_coco.bin \
  --model.load_ckpt_class_head False \
  --trainer.devices 1 \
  --trainer.gradient_clip_val 0.1 \
  --trainer.logger '{"class_path":"lightning.pytorch.loggers.CSVLogger","init_args":{"save_dir":"/content/drive/MyDrive/finetuned_cityscapes/logs"}}' \
  --trainer.max_epochs 107 \
  --data.batch_size 1 \
  --data.num_workers 2 \
  --model.lr 2e-5 \
  --model.llrd 0.9 \
  --model.network.num_q 200 \
  --model.img_size [640,640] \
  --data.img_size [640,640] \
  --trainer.default_root_dir /content/drive/MyDrive/finetuned_cityscapes

2026-05-30 15:27:41.540348: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Seed set to 0
INFO:root:Loaded 195 keys
Using 16bit Automatic Mixed Precision (AMP)
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loading `train_dataloader` to estimate number of stepping batches.

   | Name                     | Type                        | Params | Mode 
----------------------------------------------------------------------------------
0  | network                  | EoMT                